Teste V2 - YOLO + ResNet

In [ ]:
# ===== IMPORTS =====
import sys
import os
sys.path.append(os.path.abspath("../"))

import cv2
import time
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

from src.core.detector import Detector
from src.core.processor import DetectionProcessor
from src.core.visualizer import Visualizer
from src.core.classifier_v2 import ResNetClassifier
from src.core.pipeline_v2 import VisionPipelineV2


# ===== INIT =====
# V1: detector = Detector() + VisionPipeline(detector, processor, visualizer)
# V2: adiciona a ResNet como segundo estagio, rodando sobre o crop de cada
# deteccao-alvo do YOLO (classificacao ImageNet + feature extraction)
detector = Detector()
classifier = ResNetClassifier(backbone="resnet18", topk=3)

processor = DetectionProcessor(
    target_classes=["cell phone"],
    ignore_classes=["person"]
)

visualizer = Visualizer()
pipeline = VisionPipelineV2(detector, processor, visualizer, classifier)

cap = cv2.VideoCapture(0)

os.makedirs("outputs", exist_ok=True)

last_detected = []


## Loop com classificação ResNet

Mesmo loop de captura da V1, mas agora cada objeto-alvo detectado pelo YOLO também é classificado pela ResNet (top-3 categorias ImageNet), imprimindo a classificação refinada ao lado do label bruto do COCO.

In [ ]:
# ===== LOOP CONTROLADO =====
try:
    for _ in range(10000):  # limite grande, evita loop infinito travado

        ret, frame = cap.read()
        if not ret:
            break

        processed, detections, targets = pipeline.run(frame)

        current_detected = [d["label"] for d in targets]

        if current_detected and current_detected != last_detected:
            filename = f"outputs/detect_{int(time.time())}.jpg"
            cv2.imwrite(filename, processed)

            print(f"Novo detectado (YOLO): {current_detected} -> {filename}")
            for t in targets:
                print(f"  ResNet top-3 para '{t['label']}': {t.get('resnet_classification')}")

            last_detected = current_detected

        # ===== VISUAL =====
        raw_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        proc_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

        clear_output(wait=True)

        fig, ax = plt.subplots(1, 2, figsize=(10,5))

        ax[0].imshow(raw_rgb)
        ax[0].set_title("Camera Raw")
        ax[0].axis("off")

        ax[1].imshow(proc_rgb)
        ax[1].set_title("Detection + ResNet")
        ax[1].axis("off")

        display(fig)

        time.sleep(0.03)

except KeyboardInterrupt:
    print("Interrompido pelo usuario")

finally:
    cap.release()
    print("Camera liberada")


## Nota

A ResNet aqui roda em modo pré-treinado (ImageNet), sem fine-tuning no domínio do projeto. `ResNetClassifier.extract_features` fica disponível para uso futuro em tarefas de similaridade/embeddings entre objetos detectados.